# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and perform initial data processing on the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² dataset's metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata via the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List all available record sets, fields, and their `@id`s. This is crucial for referencing entities precisely in further processing and extraction.

In [ ]:
# Browse the available record sets and fields using their @id for reference

print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    print()

# Store available record set @id's for later use
record_set_ids = [record_set.id for record_set in dataset.record_sets]

## 3. Data Extraction
Load data from each record set into a `pandas` DataFrame. Use the record set and field `@id`s identified above to reference relevant data.

**Note:** If record set IDs are empty, this step will be illustrative based on available schema.

In [ ]:
# Extract data from all record sets (by @id)
dfs = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Loaded DataFrame for record set: {record_set_id} [Rows: {len(df)}]")

    # Show columns of the first record set if present
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in record set '{first_rs_id}':")
    print(dfs[first_rs_id].columns.tolist())
    dfs[first_rs_id].head()
else:
    print("No record sets found in this dataset schema. Check the dataset specification or contact the provider.")

## 4. Exploratory Data Analysis (EDA)
Apply simple data filtering, normalization, and grouping by referencing field `@id`. Modify this section as appropriate for the schema and data.

In [ ]:
# Select a numeric field for analysis by its @id

if record_set_ids:
    first_rs_id = record_set_ids[0]
    df = dfs[first_rs_id]

    # Pick a field to illustrate; in practice, replace with an actual numeric @id
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]

    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # Example: use first available numeric field
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() else 0
        
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field
        other_columns = [col for col in df.columns if col != numeric_field_id]
        group_field_id = other_columns[0] if other_columns else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nAverage '{numeric_field_id}' grouped by '{group_field_id}':")
            print(grouped_df.head())
    else:
        print("No numeric fields were found in this record set. Please inspect the DataFrame columns above.")
else:
    print("Cannot perform EDA without record sets.")

## 5. Visualization
Generate quick plots for numeric fields or relationships between fields (referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and possible_numeric_fields:
    plt.figure(figsize=(6, 4))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f"Histogram of '{numeric_field_id}'")
    plt.show()

    if group_field_id:
        grouped_df.plot(x=group_field_id, y=numeric_field_id, kind='bar', legend=False)
        plt.ylabel(f"Average {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.title(f"Average '{numeric_field_id}' by '{group_field_id}'")
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and inspect the FAIR² dataset defined by its Croissant schema, listed schema entities by their `@id`, and demonstrated basic extraction and exploratory analysis using those IDs. This approach ensures robust, schema-aligned processing for FAIR datasets.

You can further extend the notebook by referencing additional schema fields, joining across record sets (by `@id` or logical relationships), or applying advanced ML preprocessing workflows.